In [1]:
# claude is trying to see how late is jma 15 vs jma 10
import numpy as np
import pandas as pd

In [2]:
STAGE0_TAG = 'mnq-6T-3S-9-12am-RSX-vol2'
BARS_PATH = f"stage-0/{STAGE0_TAG}_bars.pqt"

bars = pd.read_parquet(BARS_PATH)

In [6]:
SF = "data/mnq-6t-3s-vol2.pqt"
src = pd.read_parquet(SF)

sess_lo = pd.Timestamp("09:00").time()
sess_hi = pd.Timestamp("12:00").time()

src = src[(src["timestamp"].dt.time >= sess_lo) & (src["timestamp"].dt.time < sess_hi)]

In [7]:
b = bars.merge(src[["timestamp", "rawLast"]], on="timestamp")     # raw close in prod1
rows = []
for (d, lid), g in b[~b.warm].groupby(["date", "leg_id"]):
    if len(g) < 2: continue
    px = g["rawLast"].to_numpy(); dirn = g["jma_leg_dir"].iloc[0]
    ext = px.argmax() if dirn > 0 else px.argmin()
    rows.append((len(g), abs(px[ext] - px[0]), abs(px[-1] - px[0]), len(g) - 1 - ext))
s = pd.DataFrame(rows, columns=["bars", "pts_to_extreme", "pts_at_signal", "lag_bars"])
print(s.median(), "\nsegments/day:", len(s) / b.date.nunique())

bars              8.00
pts_to_extreme    2.25
pts_at_signal     2.50
lag_bars          3.00
dtype: float64 
segments/day: 356.6384083044983
